In [1]:
# 导入工具包
import numpy as np
import cv2

def cv_show(name, img):
	cv2.imshow(name, img)
	cv2.waitKey(120)


def order_points(pts):
	# 一共4个坐标点
	rect = np.zeros((4, 2), dtype = "float32")

	# 按顺序找到对应坐标0123分别是 左上，右上，右下，左下
	# 计算左上，右下
	s = pts.sum(axis = 1)
	rect[0] = pts[np.argmin(s)]
	rect[2] = pts[np.argmax(s)]

	# 计算右上和左下
	diff = np.diff(pts, axis = 1)
	rect[1] = pts[np.argmin(diff)]
	rect[3] = pts[np.argmax(diff)]

	return rect

def four_point_transform(image, pts):
	# 获取输入坐标点
	rect = order_points(pts)
	(tl, tr, br, bl) = rect

	# 计算输入的w和h值
	widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
	widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
	maxWidth = max(int(widthA), int(widthB))

	heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
	heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
	maxHeight = max(int(heightA), int(heightB))

	# 变换后对应坐标位置
	dst = np.array([
		[0, 0],
		[maxWidth - 1, 0],
		[maxWidth - 1, maxHeight - 1],
		[0, maxHeight - 1]], dtype = "float32")

	# 计算变换矩阵
	M = cv2.getPerspectiveTransform(rect, dst)
	warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))

	# 返回变换后结果
	return warped

def resize(image, width=None, height=None, inter=cv2.INTER_AREA):
	dim = None
	(h, w) = image.shape[:2]
	if width is None and height is None:
		return image
	if width is None:
		r = height / float(h)
		dim = (int(w * r), height)
	else:
		r = width / float(w)
		dim = (width, int(h * r))
	resized = cv2.resize(image, dim, interpolation=inter)
	return resized

# 读取输入

import cv2

cap = cv2.VideoCapture(0)#确保摄像头是可以启动的状态。
if not cap.isOpened():#打开失败
	print("Cannot open camera")
	exit()

while True:
	flag = 0 #用于标识 当前是否检测到文档
	ret, image = cap.read() #如果正确读取帧，ret为True
	orig = image.copy()
	if not ret:#读取失败，则退出循环
		print("不能读取摄像头")
		break#
	cv_show("image", image)

	gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)# 图像处理-转换为灰度图
	# 预处理
	gray = cv2.GaussianBlur(gray, (5, 5), 0)  # 高斯滤波
	edged = cv2.Canny(gray, 75, 200)#只是得到图片的边缘，不会内部的 ，

	# 轮廓检测
	cnts = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[1]

	cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:3]
	image_contours = cv2.drawContours(image,cnts,-1, (0, 255, 0), 2)
	cv_show("image_contours", image_contours)

	# 遍历轮廓
	for c in cnts:
		# 计算轮廓近似
		peri = cv2.arcLength(c, True)
		# C表示输入的点集
		# epsilon表示从原始轮廓到近似轮廓的最大距离，它是一个准确度参数
		# True表示封闭的
		approx = cv2.approxPolyDP(c, 0.05 * peri, True)  # 轮廓近似
		area = cv2.contourArea(approx)
		# 4个点的时候就拿出来
		if area > 20000  and len(approx) == 4:
			screenCnt = approx
			flag = 1
			print(peri, area)
			print('检测到文档')
			break

	if flag == 1:
		# 展示结果
		# print("STEP 2: 获取轮廓")
		image_contours = cv2.drawContours(image, [screenCnt], 0, (0, 255, 0), 2)
		cv_show("image", image_contours)

		# 透视变换
		warped = four_point_transform(orig, screenCnt.reshape(4, 2))
		cv_show("warped", warped)

		# 二值处理
		warped = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
		ref = cv2.threshold(warped, 220, 255, cv2.THRESH_BINARY)[1]
		cv_show("ref", ref)

cap.release()  #释放捕获器
cv2.destroyAllWindows() #关闭图像窗口


Cannot open camera


AttributeError: 'NoneType' object has no attribute 'copy'